# 01 - Preparación del Dataset WLASL

## Objetivo

Convertir los videos MP4 de WLASL en secuencias de frames organizadas para entrenar modelos.

**NOTA**: Por recomendación del tutor, empezamos con **30 palabras** (WLASL30) para validar el pipeline antes de escalar.

## Qué tenemos (dataset de Kaggle)

```
kaggle_wlasl/
├── videos/              ← Videos MP4 (12,000+)
├── nslt_100.json        ← Info de WLASL100 (base para filtrar)
├── nslt_300.json        ← Info de WLASL300
├── wlasl_class_list.txt ← Mapeo: índice → palabra
└── ...
```

## Qué vamos a crear

```
processed/wlasl30/
├── train/
│   ├── book/
│   │   ├── 05237/
│   │   │   ├── 0000.jpg
│   │   │   ├── 0001.jpg
│   │   │   └── ...
│   │   └── 05238/
│   └── drink/
├── val/
├── test/
└── metadata.json
```

---
## PASO 1: Configuración

In [1]:
import os
import json
import cv2
from pathlib import Path
from tqdm import tqdm

# Paths
PROJECT_ROOT = Path("../../").resolve()
KAGGLE_DIR = PROJECT_ROOT / "06_datasets" / "wlasl" / "raw" / "kaggle_wlasl"
PROCESSED_DIR = PROJECT_ROOT / "06_datasets" / "wlasl" / "processed"

# Configuración
# Usamos nslt_100.json como base y filtramos a las N_WORDS más frecuentes
N_WORDS = 30  # <-- CAMBIAR AQUÍ para usar más/menos palabras
SUBSET = f'wlasl{N_WORDS}'  # wlasl30, wlasl50, etc.
BASE_ANNOTATIONS = 'nslt_100.json'  # Usamos WLASL100 como base y filtramos
FRAME_SIZE = (224, 224)  # Tamaño de frames para CNN
MIN_FRAMES = 8  # Mínimo frames requeridos

print(f"Dataset Kaggle: {KAGGLE_DIR}")
print(f"Salida: {PROCESSED_DIR}")
print(f"Subset: {SUBSET} ({N_WORDS} palabras más frecuentes)")

Dataset Kaggle: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/raw/kaggle_wlasl
Salida: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/processed
Subset: wlasl30 (30 palabras más frecuentes)


In [2]:
# Verificar que tenemos los archivos necesarios
videos_dir = KAGGLE_DIR / "videos"
class_list_file = KAGGLE_DIR / "wlasl_class_list.txt"
nslt_file = KAGGLE_DIR / BASE_ANNOTATIONS  # Siempre usamos nslt_100.json como base

print("Verificando archivos...")
print(f"  Videos: {videos_dir.exists()} ({len(list(videos_dir.glob('*.mp4')))} archivos)")
print(f"  Lista de clases: {class_list_file.exists()}")
print(f"  Anotaciones base: {nslt_file.exists()}")

Verificando archivos...
  Videos: True (11980 archivos)
  Lista de clases: True
  Anotaciones base: True


---
## PASO 2: Descargar dataset de Kaggle (si no existe)

Descarga ~5GB que contiene todo WLASL (2000 palabras en videos MP4).

In [3]:
import shutil

# Configurar Kaggle (copia tu kaggle.json a ~/.kaggle/)
kaggle_json_src = PROJECT_ROOT / "utils" / "kaggle.json"
kaggle_json_dst = Path.home() / ".kaggle" / "kaggle.json"

if kaggle_json_src.exists() and not kaggle_json_dst.exists():
    kaggle_json_dst.parent.mkdir(exist_ok=True)
    shutil.copy(kaggle_json_src, kaggle_json_dst)
    os.chmod(kaggle_json_dst, 0o600)
    print("Kaggle configurado")
elif kaggle_json_dst.exists():
    print("Kaggle ya configurado")
else:
    print("No se encontró kaggle.json en utils/")

Kaggle ya configurado


In [4]:
# Descargar dataset si no existe
RAW_DIR = PROJECT_ROOT / "06_datasets" / "wlasl" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

zip_path = RAW_DIR / "wlasl-processed.zip"
extracted_dir = RAW_DIR / "kaggle_wlasl"

if extracted_dir.exists() and (extracted_dir / "videos").exists():
    print("Dataset ya descargado y extraído")
else:
    import kaggle
    
    if not zip_path.exists():
        print("Descargando dataset de Kaggle (~5GB)...")
        print("Esto puede tardar 10-30 minutos dependiendo de tu conexión.\n")
        print("Dataset URL: https://www.kaggle.com/datasets/risangbaskoro/wlasl-processed\n")
        
        kaggle.api.dataset_download_files(
            dataset="risangbaskoro/wlasl-processed",
            path=str(RAW_DIR),
            unzip=False
        )
        print("Descarga completada")
    
    # Extraer ZIP
    if zip_path.exists():
        import zipfile
        print("Extrayendo ZIP...")
        extracted_dir.mkdir(exist_ok=True)
        
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extracted_dir)
        
        print("Extracción completada")

Dataset ya descargado y extraído


---
## PASO 3: Cargar anotaciones

In [5]:
# Cargar mapeo: índice -> palabra
idx_to_word = {}
with open(class_list_file, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            idx, word = parts
            idx_to_word[int(idx)] = word

print(f"Total palabras en vocabulario: {len(idx_to_word)}")
print(f"\nPrimeras 10:")
for i in range(10):
    print(f"  {i}: {idx_to_word.get(i, '?')}")

Total palabras en vocabulario: 2000

Primeras 10:
  0: book
  1: drink
  2: computer
  3: before
  4: chair
  5: go
  6: clothes
  7: who
  8: candy
  9: cousin


In [6]:
# Cargar anotaciones del subset
# Formato: {video_id: {"subset": "train", "action": [class_idx, 1, num_frames]}}
with open(nslt_file, 'r') as f:
    annotations = json.load(f)

print(f"Videos en {SUBSET}: {len(annotations)}")

# Ejemplo
ejemplo_id = list(annotations.keys())[0]
ejemplo = annotations[ejemplo_id]
print(f"\nEjemplo:")
print(f"  Video ID: {ejemplo_id}")
print(f"  Split: {ejemplo['subset']}")
print(f"  Clase: {ejemplo['action'][0]} -> {idx_to_word.get(ejemplo['action'][0], '?')}")
print(f"  Frames: {ejemplo['action'][2]}")

Videos en wlasl30: 2038

Ejemplo:
  Video ID: 05237
  Split: train
  Clase: 77 -> basketball
  Frames: 55


In [7]:
# Contar distribución por split
split_counts = {'train': 0, 'val': 0, 'test': 0}
for vid, info in annotations.items():
    split_counts[info['subset']] += 1

print("Distribución por split:")
for split, count in split_counts.items():
    print(f"  {split}: {count}")

Distribución por split:
  train: 1442
  val: 338
  test: 258


In [8]:
# Contar por palabra
from collections import defaultdict

word_counts = defaultdict(int)
for vid, info in annotations.items():
    class_idx = info['action'][0]
    word = idx_to_word.get(class_idx, f'unknown_{class_idx}')
    word_counts[word] += 1

print(f"Palabras únicas: {len(word_counts)}")
print(f"\nTop 10 palabras con más videos:")
for word, count in sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {word}: {count}")

Palabras únicas: 100

Top 10 palabras con más videos:
  book: 40
  drink: 35
  computer: 30
  before: 26
  go: 26
  chair: 26
  who: 25
  clothes: 25
  candy: 24
  cousin: 23


---
## PASO 3.5: Filtrar a las N palabras más frecuentes

Según recomendación del tutor, empezamos con un subconjunto pequeño (30 palabras) para validar el pipeline rápidamente.

In [9]:
# FILTRAR A LAS N_WORDS PALABRAS MÁS FRECUENTES
# Ordenamos las palabras por cantidad de videos y tomamos las top N_WORDS

top_words = [word for word, count in sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:N_WORDS]]

print(f"Filtrando a las {N_WORDS} palabras más frecuentes:")
print(f"  {', '.join(top_words[:10])}{'...' if N_WORDS > 10 else ''}")

# Crear mapeo inverso: palabra -> índice original
word_to_original_idx = {v: k for k, v in idx_to_word.items()}

# Filtrar anotaciones: solo mantener videos de las top N_WORDS palabras
filtered_annotations = {}
for vid, info in annotations.items():
    class_idx = info['action'][0]
    word = idx_to_word.get(class_idx, '')
    if word in top_words:
        filtered_annotations[vid] = info

# Reemplazar annotations con las filtradas
annotations = filtered_annotations

print(f"\nVideos después de filtrar: {len(annotations)}")

# Verificar distribución
split_counts = {'train': 0, 'val': 0, 'test': 0}
for vid, info in annotations.items():
    split_counts[info['subset']] += 1

print(f"  - Train: {split_counts['train']}")
print(f"  - Val: {split_counts['val']}")
print(f"  - Test: {split_counts['test']}")

Filtrando a las 30 palabras más frecuentes:
  book, drink, computer, before, go, chair, who, clothes, candy, cousin...

Videos después de filtrar: 709
  - Train: 497
  - Val: 117
  - Test: 95


In [10]:
## PASO 4: Extraer frames de los videos
def extract_frames_from_video(video_path, output_dir, frame_size=(224, 224)):
    """
    Extrae todos los frames de un video y los guarda como imágenes.
    
    Args:
        video_path: Path al archivo MP4
        output_dir: Directorio donde guardar los frames
        frame_size: Tamaño al que redimensionar los frames
    
    Returns:
        int: Número de frames extraídos
    """
    video_path = Path(video_path)
    output_dir = Path(output_dir)
    
    if not video_path.exists():
        return 0
    
    # Abrir video
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0
    
    # Crear directorio de salida
    output_dir.mkdir(parents=True, exist_ok=True)
    
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Redimensionar
        frame = cv2.resize(frame, frame_size)
        
        # Guardar
        frame_path = output_dir / f"{frame_count:04d}.jpg"
        cv2.imwrite(str(frame_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        
        frame_count += 1
    
    cap.release()
    return frame_count

In [11]:
# Test con un video
test_vid = list(annotations.keys())[0]
test_video_path = videos_dir / f"{test_vid}.mp4"
test_output = PROCESSED_DIR / "test_extraction"

print(f"Test: extrayendo frames de {test_vid}.mp4")
n_frames = extract_frames_from_video(test_video_path, test_output, FRAME_SIZE)
print(f"Frames extraídos: {n_frames}")

# Limpiar test
import shutil
if test_output.exists():
    shutil.rmtree(test_output)
    print("Test limpiado")

Test: extrayendo frames de 69422.mp4
Frames extraídos: 59
Test limpiado


---
## PASO 5: Procesar todos los videos

In [12]:
# Directorio de salida
output_dir = PROCESSED_DIR / SUBSET

# Crear directorios de splits
for split in ['train', 'val', 'test']:
    (output_dir / split).mkdir(parents=True, exist_ok=True)

print(f"Directorio de salida: {output_dir}")
print(f"Procesando {len(annotations)} videos...\n")

Directorio de salida: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/processed/wlasl30
Procesando 709 videos...



In [13]:
# Procesar todos los videos
processed_sequences = []
errors = []
skipped = []

for video_id, info in tqdm(annotations.items(), desc="Extrayendo frames"):
    try:
        # Info del video
        split = info['subset']
        class_idx = info['action'][0]
        word = idx_to_word.get(class_idx, f'unknown_{class_idx}')
        
        # Paths
        video_path = videos_dir / f"{video_id}.mp4"
        frames_dir = output_dir / split / word / video_id
        
        # Si ya procesado, solo contar frames
        if frames_dir.exists() and any(frames_dir.glob('*.jpg')):
            n_frames = len(list(frames_dir.glob('*.jpg')))
        else:
            # Verificar que existe el video
            if not video_path.exists():
                errors.append(f"{video_id}: video no encontrado")
                continue
            
            # Extraer frames
            n_frames = extract_frames_from_video(video_path, frames_dir, FRAME_SIZE)
        
        # Verificar mínimo de frames
        if n_frames >= MIN_FRAMES:
            processed_sequences.append({
                'video_id': video_id,
                'gloss': word,
                'split': split,
                'num_frames': n_frames,
                'path': f"{split}/{word}/{video_id}"
            })
        else:
            skipped.append(f"{video_id}: solo {n_frames} frames")
            # Eliminar carpeta si tiene muy pocos frames
            if frames_dir.exists():
                shutil.rmtree(frames_dir)
                
    except Exception as e:
        errors.append(f"{video_id}: {str(e)}")

print(f"\n{'='*50}")
print(f"RESUMEN")
print(f"{'='*50}")
print(f"✓ Procesados correctamente: {len(processed_sequences)}")
print(f"⚠ Saltados (pocos frames): {len(skipped)}")
print(f"✗ Errores: {len(errors)}")

Extrayendo frames: 100%|██████████| 709/709 [00:39<00:00, 17.85it/s]


RESUMEN
✓ Procesados correctamente: 337
⚠ Saltados (pocos frames): 0
✗ Errores: 372


In [14]:
# Mostrar algunos errores si los hay
if errors:
    print("Primeros 10 errores:")
    for e in errors[:10]:
        print(f"  {e}")

Primeros 10 errores:
  64298: video no encontrado
  64299: video no encontrado
  64296: video no encontrado
  64297: video no encontrado
  64294: video no encontrado
  64295: video no encontrado
  64290: video no encontrado
  05735: video no encontrado
  05736: video no encontrado
  05737: video no encontrado


---
## PASO 6: Crear metadata.json

In [15]:
# Estadísticas
all_words = sorted(set(s['gloss'] for s in processed_sequences))
frame_counts = [s['num_frames'] for s in processed_sequences]

metadata = {
    'name': SUBSET,
    'description': f'WLASL {SUBSET} - Word-Level American Sign Language',
    'frame_size': list(FRAME_SIZE),
    
    'statistics': {
        'total_sequences': len(processed_sequences),
        'total_words': len(all_words),
        'train': len([s for s in processed_sequences if s['split'] == 'train']),
        'val': len([s for s in processed_sequences if s['split'] == 'val']),
        'test': len([s for s in processed_sequences if s['split'] == 'test']),
        'frames_min': min(frame_counts) if frame_counts else 0,
        'frames_max': max(frame_counts) if frame_counts else 0,
        'frames_mean': round(sum(frame_counts) / len(frame_counts), 1) if frame_counts else 0
    },
    
    'glosses': all_words,
    'gloss_to_idx': {w: i for i, w in enumerate(all_words)},
    'idx_to_gloss': {i: w for i, w in enumerate(all_words)},
    
    'sequences': processed_sequences
}

# Guardar
metadata_path = output_dir / 'metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata guardada: {metadata_path}")
print(f"\n{'='*50}")
print("ESTADÍSTICAS DEL DATASET")
print(f"{'='*50}")
print(f"Palabras: {metadata['statistics']['total_words']}")
print(f"Secuencias totales: {metadata['statistics']['total_sequences']}")
print(f"  - Train: {metadata['statistics']['train']}")
print(f"  - Val: {metadata['statistics']['val']}")
print(f"  - Test: {metadata['statistics']['test']}")
print(f"Frames por secuencia: {metadata['statistics']['frames_min']}-{metadata['statistics']['frames_max']} (media: {metadata['statistics']['frames_mean']})")

✓ Metadata guardada: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/processed/wlasl30/metadata.json

ESTADÍSTICAS DEL DATASET
Palabras: 30
Secuencias totales: 337
  - Train: 245
  - Val: 57
  - Test: 35
Frames por secuencia: 20-195 (media: 63.7)


---
## PASO 7: Verificación

In [16]:
# Ver estructura creada
print("Estructura del dataset procesado:\n")

for split in ['train', 'val', 'test']:
    split_dir = output_dir / split
    if split_dir.exists():
        words = [d for d in split_dir.iterdir() if d.is_dir()]
        total_videos = sum(len([v for v in w.iterdir() if v.is_dir()]) for w in words)
        print(f"📁 {split}/")
        print(f"   {len(words)} palabras, {total_videos} videos")
        
        # Mostrar algunas palabras
        for word_dir in sorted(words)[:3]:
            n_videos = len([v for v in word_dir.iterdir() if v.is_dir()])
            print(f"   └── {word_dir.name}/ ({n_videos} videos)")

Estructura del dataset procesado:

📁 train/
   30 palabras, 245 videos
   └── all/ (7 videos)
   └── before/ (12 videos)
   └── black/ (8 videos)
📁 val/
   27 palabras, 57 videos
   └── before/ (1 videos)
   └── black/ (2 videos)
   └── book/ (1 videos)
📁 test/
   23 palabras, 35 videos
   └── all/ (1 videos)
   └── before/ (3 videos)
   └── candy/ (3 videos)


---
## Resumen

### Qué hicimos:
1. Descargamos WLASL de Kaggle (~5GB de videos MP4)
2. Cargamos las anotaciones de WLASL100 como base
3. **Filtramos a las N_WORDS palabras más frecuentes** (30 por defecto)
4. Extrajimos frames de cada video MP4 usando OpenCV
5. Organizamos en estructura: `{split}/{palabra}/{video_id}/frames`
6. Creamos `metadata.json` con toda la información

### El metadata.json contiene:
- `glosses`: Lista de palabras
- `gloss_to_idx`: Mapeo palabra → número
- `sequences`: Info de cada video (path, palabra, split, num_frames)

### Para cambiar el número de palabras:
Modifica `N_WORDS` en el PASO 1 y re-ejecuta el notebook.

### Próximo paso:
Notebook `02_cnn_lstm_baseline.ipynb` para crear el DataLoader y entrenar el modelo

---
## Resumen

### Qué hicimos:
1. Descargamos WLASL de Kaggle (~5GB de videos MP4)
2. Cargamos las anotaciones de WLASL100 (qué videos usar y sus etiquetas)
3. Extrajimos frames de cada video MP4 usando OpenCV
4. Organizamos en estructura: `{split}/{palabra}/{video_id}/frames`
5. Creamos `metadata.json` con toda la información

### El metadata.json contiene:
- `glosses`: Lista de palabras
- `gloss_to_idx`: Mapeo palabra → número
- `sequences`: Info de cada video (path, palabra, split, num_frames)

### Próximo paso:
Notebook `02_cnn_lstm_baseline.ipynb` para crear el DataLoader y entrenar el modelo